# RSP Setup Check

Use this notebook first on Rubin Science Platform. It verifies the local checkout, persistent storage paths, package imports, and a tiny LSST-only ANTARES probe before any long backfill.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    candidate = Path.home() / 'notebooks' / 'ANTARES_Analysis'
    if (candidate / 'src').exists():
        PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root : {PROJECT_ROOT}')
print(f'HOME         : {os.getenv("HOME")}')
print(f'USER         : {os.getenv("USER") or os.getenv("JUPYTERHUB_USER")}')
print(f'SCRATCH_DIR  : {os.getenv("SCRATCH_DIR", "not set")}')
print(f'Project area : /project/{os.getenv("USER") or os.getenv("JUPYTERHUB_USER") or "unknown_user"}')

In [ ]:
import pandas as pd
import pyarrow
from antares_client.search import search as antares_search

from src import config, history, query

print('Imports complete.')
print(f'pandas  : {pd.__version__}')
print(f'pyarrow : {pyarrow.__version__}')
config.print_config_summary()

In [ ]:
DATA_ROOT = Path(config.HISTORY_DATA_ROOT)
LSST_DATA_ROOT = history.survey_data_root(DATA_ROOT)
SCRATCH_ROOT = Path(os.getenv('SCRATCH_DIR', f'/scratch/{os.getenv("USER") or os.getenv("JUPYTERHUB_USER") or "unknown_user"}')) / 'ANTARES_Analysis'

for path in [
    LSST_DATA_ROOT / 'nightly',
    LSST_DATA_ROOT / 'cumulative',
    DATA_ROOT / 'data' / 'ztf_archive_do_not_use_for_lsst',
    DATA_ROOT / 'figures',
    DATA_ROOT / 'logs',
    SCRATCH_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)
    print(f'OK: {path}')

print(f'Persistent LSST store: {LSST_DATA_ROOT}')
print(f'Temporary cache root : {SCRATCH_ROOT}')


In [ ]:
probe = query.query_range(
    label='RSP LSST-only probe',
    mjd_min=config.LSST_HISTORY_START_MJD,
    mjd_max=config.LSST_HISTORY_START_MJD + 1,
    n_samples=5,
    tag=config.QUERY_TAG,
    seed=None,
    verbose=True,
    lsst_only=True,
)

counts = query.lsst_identifier_counts(probe)
print(counts)
if not probe.empty:
    assert counts['lsst_identifier_count'] == len(probe), 'Probe returned non-LSST loci.'
    display_cols = [col for col in ['locus_id', 'ra', 'dec', 'newest_alert_observation_time', 'survey', 'ztf_object_id'] if col in probe.columns]
    display(probe[display_cols].head())
else:
    print('Probe returned 0 rows. Try a later MJD window before running backfill.')